In [1]:
#Imports and load the data

import pandas as pd
import numpy as np
import sys
import os

sys.path.append('../')
from src.data_loader import load_cement_data

import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX

df = load_cement_data()
df.shape

(32880, 13)

In [2]:
#Keep one site and set up the dates
site_id = 'SITE_001'
site_df = df[df['site_id'] == site_id].copy()

site_df.set_index('date', inplace=True)
site_df = site_df.sort_index()
site_df = site_df.asfreq('D')

site_df.shape

(1096, 12)

In [3]:
#Target and inputs

y = site_df['consumed_tonnes']
exog = site_df[['planned_pour_tonnes', 'rain_mm', 'avg_temp_c']]

In [4]:
#Split into training and test periods
split_index = int(len(site_df) * 0.8)

X_train, X_test = exog.iloc[:split_index], exog.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Training:", y_train.index.min().date(), "to", y_train.index.max().date(), "-", len(y_train), "days")
print("Testing: ", y_test.index.min().date(), "to", y_test.index.max().date(), "-", len(y_test), "days")

Training: 2022-01-01 to 2024-05-25 - 876 days
Testing:  2024-05-26 to 2024-12-31 - 220 days


In [5]:
#Building and fit the SARIMAX model

model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False)
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                   consumed_tonnes   No. Observations:                  876
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3381.155
Date:                           Mon, 16 Nov 2026   AIC                           6778.310
Time:                                   14:35:48   BIC                           6816.421
Sample:                               01-01-2022   HQIC                          6792.895
                                    - 05-25-2024                                         
Covariance Type:                             opg                                         
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
planned_pour_tonnes     0.5504      0.039     14.239      0.000       0.475       0.626
rain_mm                -0.8606      0.067    -12.835      0.000      -0.992      -0.729
avg_temp_c              0.0497      0.052      0.962      0.336      -0.052       0.151
ar.L1                   0.0849      0.034      2.473      0.013       0.018       0.152
ma.L1                  -0.9970      0.004   -225.112      0.000      -1.006      -0.988
ar.S.L7                -0.0430      0.592     -0.073      0.942      -1.202       1.116
ma.S.L7                 0.0188      0.591      0.032      0.975      -1.140       1.177
sigma2                143.4096      8.812     16.275      0.000     126.139     160.681
===================================================================================
Ljung-Box (L1) (Q):                   0.00   Jarque-Bera (JB):                35.97
Prob(Q):                              1.00   Prob(JB):                         0.00
Heteroskedasticity (H):               0.93   Skew:                            -0.43
Prob(H) (two-sided):                  0.55   Kurtosis:                         2.48
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [6]:
#Forecast the test period

sarimax_forecast = results.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
sarimax_forecast.head()

2024-05-26    31.807181
2024-05-27    39.369151
2024-05-28    10.882286
2024-05-29    31.873212
2024-05-30    32.608035
Freq: D, Name: predicted_mean, dtype: float64

In [7]:
#Measure SARIMAX

from src.metrics import print_metrics

print_metrics(y_test, sarimax_forecast, 'SARIMAX')

SARIMAX - MAPE: 36.17%, RMSE: 11.73 tonnes


In [8]:
#AdaBoost

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=4),
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)
ada.fit(X_train, y_train)

ada_forecast = ada.predict(X_test)
print_metrics(y_test, ada_forecast, 'AdaBoost')

AdaBoost - MAPE: 33.21%, RMSE: 9.73 tonnes


In [10]:
#Chart the forecasts against actual consumption

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=y_test.index, y=y_test, mode='lines',
                         name='Actual', line=dict(color='black')))
fig.add_trace(go.Scatter(x=y_test.index, y=sarimax_forecast, mode='lines',
                         name='SARIMAX forecast', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=y_test.index, y=ada_forecast, mode='lines',
                         name='AdaBoost forecast', line=dict(color='red')))

fig.update_layout(
    title=f'Forecast vs Actual Consumption - {site_id}',
    xaxis_title='Date',
    yaxis_title='Cement Consumed (tonnes)',
    legend=dict(x=0, y=1),
    template='plotly_white',
    width=1000,
    height=450
)

fig.show()

In [11]:
model_nw = SARIMAX(y_train, exog=X_train, order=(1, 1, 1),
                   enforce_stationarity=False, enforce_invertibility=False)
results_nw = model_nw.fit(disp=False)

sarimax_nw_forecast = results_nw.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
print_metrics(y_test, sarimax_nw_forecast, 'SARIMAX (no weekly)')

SARIMAX (no weekly) - MAPE: 36.15%, RMSE: 11.73 tonnes
